In [1]:
import sys, os
d = os.getcwd()
while not os.path.isdir(os.path.join(d, "data")) and d != os.path.dirname(d):
    d = os.path.dirname(d)          # climb until we find the folder containing 'data'
sys.path.insert(0, d)
from data.src.api.statsapi import Statsapi
import os
from dotenv import load_dotenv, find_dotenv
import json

load_dotenv(find_dotenv(), override=True)                          # reads .env into the environment
key = os.getenv("STATS_API_KEY")

stats = Statsapi(key)

In [2]:
matches = stats.get_games_today()

In [3]:
from run_preds import run_preds

In [4]:
preds, models, shares, rrates, rbase, ref_table = run_preds('../data/data_logs/data_sets_clean/dixon_coles_20260624_inputs.csv', matches)

In [19]:
from modeling.models.simulate import simulate_match
from modeling.models.referee import penalty_rate

sims, sim_markets = simulate_match(
    models, shares,
    matches[1]['home_team']['id'], matches[1]['away_team']['id'],   # adjust to your real match keys
    neutral=True, n_sims=50_000,
    ref_table=ref_table, ref_id=matches[1].get('ref_id'),
    red_rates=rrates, red_base=rbase,
    pen_rate=penalty_rate(ref_table, matches[1].get('ref_id')),
    seed=1,
)
print(sim_markets["over_2.5_AND_btts"])

0.2651


In [ ]:
hg1, ag1 = sims["goals"]["1h"]      # home 1H goals array, away 1H goals array
hg2, ag2 = sims["goals"]["2h"]      # home 2H, away 2H
hgf, agf = sims["goals"]["ft"]
h2_sot, a2_sot = sims["sot"]["2h"]  # shots on target
hf_sot, af_sot = sims["sot"]["ft"]
fh, fa = sims['fouls']['ft']        # fouls ft home & away
hcd, acd = sims["cards"]["ft"]          # home & away FT card arrays
h_off, a_off = sims["offsides"]["ft"]       # home & away FT offsides arrays

In [24]:
print(hg1.mean(), ag1.mean(), hg2.mean(), ag2.mean())
import numpy as np
print("P(away has more SoT in 2H):", np.mean(hg2 > 0))


0.29068 0.46656 0.37468 0.98402
P(away has more SoT in 2H): 0.31214


In [23]:
h_off, a_off = sims["offsides"]["ft"]       # home & away FT offsides arrays
print("P(away 2+ offsides FT):", np.mean(h_off >= 2))

P(away 2+ offsides FT): 0.40728


In [27]:
hcd, acd = sims["cards"]["ft"]          # home & away FT card arrays
total = hcd + acd
print("P(4+ total cards):", np.mean(total >= 4))

P(4+ total cards): 0.6034


In [12]:
import pandas as pd

def flatten_pred(p):
    g = p["goals"]["ft"]
    return {
        "home_win": g["result"]["home"],
        "draw":     g["result"]["draw"],
        "away_win": g["result"]["away"],
        "xg_home":  g["home_exp"],
        "xg_away":  g["away_exp"],
        "o2.5":     g["over"]["2.5"],
        "btts":     g["btts"],
        "corners":  p["corners"]["ft"]["total_exp"],
        "cards":    p["cards"]["ft"]["total_exp"],
        "o3.5_cards": p["cards"]["ft"]["over"]["3.5"],
        "fouls":    p["fouls"]["ft"]["total_exp"],
        "red":      p["red_card"]["ft"],
        "pen":      p["penalty"]["ft"],
    }

view = pd.DataFrame([flatten_pred(p) for p in preds]).round(3)
view

,home_win,draw,away_win,xg_home,xg_away,o2.5,btts,corners,cards,o3.5_cards,fouls,red,pen
0,0.180,0.287,0.533,0.729,1.441,0.369,0.395,8.004,3.261,0.411,25.757,0.105,0.236
1,0.165,0.282,0.553,0.687,1.474,0.367,0.383,7.053,4.344,0.631,25.476,0.118,0.203
2,0.724,0.214,0.062,1.810,0.368,0.371,0.258,8.929,3.732,0.512,36.238,0.098,0.240
3,0.069,0.156,0.775,0.637,2.497,0.606,0.433,7.733,4.243,0.612,27.593,0.058,0.202
4,0.420,0.290,0.290,1.351,1.081,0.439,0.490,8.055,5.629,0.812,31.685,0.140,0.248
5,0.314,0.313,0.373,1.028,1.144,0.370,0.438,8.701,4.404,0.639,24.170,0.131,0.221


In [13]:
for match in matches:
    print(f'Home: {match['home_team']['name']}    Away: {match['away_team']['name']}')


Home: South Africa    Away: South Korea
Home: Czechia    Away: Mexico
Home: Morocco    Away: Haiti
Home: Scotland    Away: Brazil
Home: Bosnia & Herzegovina    Away: Qatar
Home: Switzerland    Away: Canada
